# 04 · Qué variable se usa para qué (perfil v02)

**Objetivo:** dejar por escrito, con datos, qué variables forman el **perfil de municipio**, cuáles quedan como **datos sueltos** (filtros, scoring y RAG) y por qué. Este notebook **no entrena ningún modelo**: solo recoge la decisión y comprueba la cobertura de cada variable.

**Orden:** primero 01A, 03A y 03B (dataset `sitewise_v04_candidata`). Este notebook guarda `variables_perfil_v02.csv`, que el 05 comprueba antes de empezar. La agrupación temática de los 43 indicadores y la construcción de las dimensiones están en 03A y 03B; aquí se explica la decisión final.

**Criterios (revisión técnica, Juan Miguel):**
1. Solo se agrupan variables cuyo sentido **bueno/malo es claro y consistente**.
2. El cluster final usa **como máximo 3–4 entradas** (es un techo, no un mínimo).
3. Lo que no tiene un sentido claro (población, paro, viento) no se agrupa: queda como dato suelto, filtro para el RAG o regla del scoring de cada actividad.
4. El perfil **describe** el municipio; el scoring por actividad **evalúa**; la IA **explica**. Un perfil no es una nota de atractivo ni de riesgo.

> Esta versión sustituye a la selección de las 7 variables de v01 (antiguo notebook 04 y `docs/methodology/seleccion_variables_ml.md`), que sigue en el historial de Git (commit `bc30250`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
DATASET = ROOT / 'data/processed/sitewise_v04_candidata/dataset_municipal.csv'
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 80)

d = pd.read_csv(DATASET, dtype={'cod_ine': str})
assert len(d) == 8132
print(d.shape)

(8132, 140)


## 1. Tabla de decisión

- **Perfil**: entra en los tres índices (logística, hídrica, conectividad) que usa K-Means.
- **Dato suelto**: no entra en el perfil; sirve para filtros, para el scoring de cada actividad y para que el RAG responda con el dato en bruto.
- **Sentido**: hacia dónde es "mejor". *Depende* = no hay un sentido único: lo decide la actividad o el cliente.

In [2]:
VARIABLES = [
    # variable, índice o uso, sentido, motivo
    ('suelo_industrial_cartografiado_pct', 'Perfil · logística', 'mayor, mejor', 'Suelo industrial disponible respecto al término municipal.'),
    ('dist_recta_eje_autovia_km', 'Perfil · logística', 'menor, mejor', 'Acceso por carretera.'),
    ('dist_recta_ttm_km', 'Perfil · logística', 'menor, mejor', 'Acceso a terminal de mercancías. Sin dato en Canarias y Baleares: no se imputa.'),
    ('estres_hidrico_pct', 'Perfil · hídrica', 'mayor, más presión', 'Presión sobre el agua. Los huecos se estiman con los 5 municipios cercanos y se marcan.'),
    ('agua_subterranea_mal_estado_cuantitativo_pct', 'Perfil · hídrica', 'mayor, más presión', 'Acuíferos en mal estado cuantitativo. Huecos como el anterior.'),
    ('cobertura_ftth_pct', 'Perfil · conectividad', 'mayor, mejor', 'Fibra hasta el hogar.'),
    ('cobertura_5g_pct', 'Perfil · conectividad', 'mayor, mejor', 'Cobertura 5G.'),
    ('poblacion_2025', 'Dato suelto · filtro RAG', 'depende', 'Depende del cliente; su rango enorme dominaría cualquier agrupación.'),
    ('densidad_hab_km2', 'Dato suelto', 'depende', 'Sin sentido bueno/malo único.'),
    ('variacion_poblacion_pct', 'Dato suelto', 'depende', 'Sin sentido bueno/malo único.'),
    ('paro_total', 'Dato suelto', 'depende', 'En municipios pequeños el SEPE oculta el dato; cualquier tasa depende de la población por construcción.'),
    ('viento_100m_media_m_s', 'Dato suelto · scoring', 'depende', 'Sirve a unas actividades (eólica) y no a otras.'),
    ('irradiacion_ghi_kwh_m2_anio', 'Dato suelto · scoring', 'depende', 'Sirve a unas actividades (solar) y no a otras.'),
    ('incendios_ge1ha_100km2_anio', 'Dato suelto · riesgo', 'mayor, peor', 'Cobertura parcial y sin imputar: un hueco no es riesgo cero. Se puede mostrar el dato de los municipios cercanos.'),
    ('jrc_t100_area_positiva_pct', 'Dato suelto · riesgo', 'mayor, peor', 'Cobertura parcial y sin imputar, como el anterior.'),
]
tabla = pd.DataFrame(VARIABLES, columns=['variable', 'uso', 'sentido', 'motivo'])
faltan = [v for v in tabla['variable'] if v not in d.columns]
assert not faltan, faltan
tabla['con_dato_%'] = [round(d[v].notna().mean() * 100, 1) for v in tabla['variable']]
print(tabla[['variable', 'uso', 'sentido', 'con_dato_%']].to_string(index=False))

                                    variable                      uso            sentido  con_dato_%
          suelo_industrial_cartografiado_pct       Perfil · logística       mayor, mejor       100.0
                   dist_recta_eje_autovia_km       Perfil · logística       menor, mejor        99.3
                           dist_recta_ttm_km       Perfil · logística       menor, mejor        98.0
                          estres_hidrico_pct         Perfil · hídrica mayor, más presión        99.7
agua_subterranea_mal_estado_cuantitativo_pct         Perfil · hídrica mayor, más presión        91.1
                          cobertura_ftth_pct    Perfil · conectividad       mayor, mejor       100.0
                            cobertura_5g_pct    Perfil · conectividad       mayor, mejor       100.0
                              poblacion_2025 Dato suelto · filtro RAG            depende       100.0
                            densidad_hab_km2              Dato suelto            depende   

## 2. Qué se concluye

- El perfil usa **7 variables agrupadas en 3 índices** → **3 entradas** para K-Means, por debajo del techo de 3–4.
- Las variables de riesgo natural (incendios, inundaciones) **no entran** porque su cobertura es parcial; meterlas dejaría el perfil con la mitad de los municipios (se comprueba en el notebook 05) y rellenar los huecos con 0 diría "sin riesgo" cuando solo falta el dato.
- El agua se completa con vecinos solo para el perfil y la app, siempre **marcada** como estimada (notebook 08).
- Población, paro, viento e irradiación siguen disponibles en bruto para el scoring por actividad y para el RAG.

El siguiente paso es el 05 (que lee la tabla guardada abajo): cuántos grupos (K) y qué alternativas de entrada se han probado.

In [3]:
SALIDA = ROOT / 'data/processed/sitewise_ml'
SALIDA.mkdir(parents=True, exist_ok=True)
tabla.to_csv(SALIDA / 'variables_perfil_v02.csv', index=False, encoding='utf-8', lineterminator='\n')
print('Guardado:', (SALIDA / 'variables_perfil_v02.csv').relative_to(ROOT))

Guardado: data\processed\sitewise_ml\variables_perfil_v02.csv
